### Text Summarization Case Study

### Introduction
- Text summarization is the process of automatically generating a concise and meaningful summary from a longer text
- Types of text summarization:
    - **Extractive summarization**: Selects important sentences directly from the text
    - **Abstractive summarization**: Generates new sentences that capture the essence of the text in a more natural way
- Modern GenAI models can perform high-quality abstractive summarization by understanding context and meaning
- Applications
    - Summarizing medical reports and claim documents to speed claim approval
    - Medical case summarization for underwriting
    - Summarizing doctor-patient interactions to help insurers validate medical necessity
    - Providing quick summaries of treatment plans for insurance approvals


## Problem Statement
- Clinical dialogue is an essential part of the clinical workflow. Clinical documentation is a two-step process. 
  - It first engages patients through conversation with a doctor to collect patient-specific information such as demographic information, family history of diseases, and the signs and symptoms.
  - The electronic health records (EHRs) are then generated from the conversation.
- Currently, clinical documentation is mainly done by physicians at both steps, a labor-intensive process that contributes to physician burnout.
- Extracting the useful information from the conversation between a patient and physician is a challenge, because the conversation will not have any particular structure/format (like a questionnaire).
- However, it is important to collage the required information in less words and save it for further usage.
- The task here is to summarize the conversation in a meaningful structured format with important key words, symptoms & diagnosis etc. The challenge is to use appropriate prompts to get expected outcome.

In [ ]:
%run /Users/csharm33/code/genai_dbx/Course2/.setup/learner_setup.ipynb

In [ ]:
import textwrap
import openai
import os
import json
import httpx
from dotenv import load_dotenv
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)

In [ ]:
# Authentication:
def get_access_token():
    auth = "https://api.uhg.com/oauth2/token"
    scope = "https://api.uhg.com/.default"
    grant_type = "client_credentials"


    with httpx.Client() as client:
        body = {
            "grant_type": grant_type,
            "scope": scope,
            "client_id": client_id,
            "client_secret": client_secret,
        }
        headers = {"Content-Type": "application/x-www-form-urlencoded"}
        resp = client.post(auth, headers=headers, data=body, timeout=60)
        access_token = resp.json()["access_token"]
        return access_token


load_dotenv('/Users/csharm33/code/genai_dbx/Course2/Data/vars.env')

endpoint = os.environ.get("MODEL_ENDPOINT")
model_name = os.environ.get("MODEL_NAME")
project_id = os.environ.get("PROJECT_ID")
api_version = os.environ.get("API_VERSION")
client_id = os.environ.get("CLIENT_ID")
client_secret = os.environ.get("CLIENT_SECRET")


chat_client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_version=api_version,
        azure_deployment=model_name,
        azure_ad_token=get_access_token(),
        default_headers={
            "projectId": project_id
        }
    )


In [ ]:
# We use an exponential backoff decorator in the event too 
# many users are using the model at once and rate limit is reached
@retry(wait=wait_random_exponential(min=45, max=120), stop=stop_after_attempt(6))
def query_llm(prompt_messages, max_tokens=4096, temperature=1.0, top_p=1.0):
    
    response = chat_client.chat.completions.create(
        messages=prompt_messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        model=model_name
    )

    return {'text': response.choices[0].message.content}


#### Read Doctor - Patient Conversation File

In [ ]:
# Use appropriate file path
filepath = "Data/TextSummarization/conversation_498.txt"

with open(filepath, 'r') as file:
    conversation = file.read()
print(conversation)

In [ ]:
prompt_messages = [
    {"role": "developer", "content": "You are an assistant that summarizes conversations between Doctor and Patient"},
    {"role": "user", "content": f"Please summarize the following text:\n\n{conversation}\n\nSummary:"}
]

response = query_llm(prompt_messages)

print("\n".join(textwrap.wrap(response['text'], width=80)))

In [ ]:
# print no. of words in the summary
print(f"Number of words in summary: {len(response['text'].split())}")

In [ ]:
prompt_messages = [
    {"role": "developer", "content": "You are an assistant that summarizes conversations \
      between Doctor and Patient. Make sure to provide concise summary in 100 words."},
    {"role": "user", "content": f"Please summarize the following text:\n\n{conversation}\n\nSummary:"}
]

response = query_llm(prompt_messages)
print(response['text'])

In [ ]:
# print no. of words in summary
print(f"Number of words in summary: {len(response['text'].split())}")


#### Enhanced prompt with clear instructions
- Let us now enhance our prompt by mentioning what exactly we are expecting from the summary of the conversation document.
- Some of the important instructions being given are:
  - Summary should be structured with a bullet points
  - Summary should contain around 150 words
  - Summary must capture the key point in the conversations like symptoms of the patient, diagnosis suggested by the doctor (if any), prescribed treatment (if any), and the follow-up instructions given by the doctor.


In [ ]:
prompt_messages = [
    {"role": "developer", "content": "You are an AI assistant specializing in summarizing conversations between a doctor and a patient. \
     Your task is to generate a clear, concise, and structured summary in 150 words or less. \
     Ensure that the summary captures the key points, including the patient's symptoms, \
     diagnosis, prescribed treatment, and follow-up instructions. \
     Present the summary in bullet points for easy readability"},
    {"role": "user", "content": f"Please summarize the following text:\n\n{conversation}\n\nSummary:"}
]

response = query_llm(prompt_messages)
print(response['text'])

In [ ]:
# print no. of words in summary
print(f"Number of words in summary: {len(response['text'].split())}")